# Annotation Comparison

Visualizes annotated instances from the event sequencing task. For each instance, the text is shown with the two pre-highlighted spans, followed by each annotator's response (is each span an event? if both are events: sequencing and causality ratings).

In [1]:
import json
import csv
import ast
from IPython.display import display, HTML
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

## Load Data

In [2]:
CSV_PATH = '../data/dolma_combined_final_sample_600_with_llm_summary_safeid_with_spans.csv'
RESULTS_DIR = '../annotation_output/results'
ANNOTATORS = ['adde1214', 'tejo9855', 'umasgunturi@gmail.com']

# ── Load text data ──────────────────────────────────────────────────────────
csv.field_size_limit(10**7)
with open(CSV_PATH) as f:
    reader = csv.DictReader(f)
    text_rows = {row['safe_instance_id']: row for row in reader}

print(f'Loaded {len(text_rows)} instances from CSV')

# ── Load annotation data ─────────────────────────────────────────────────────
def parse_annotation(val):
    """Extract annotation dict from instance_id_to_label_to_value entry."""
    inner = val[0][1]
    if not inner:
        return None
    parsed = json.loads(inner)
    return parsed[0] if parsed else None

annotator_data = {}  # annotator -> {inst_id -> annotation_dict}
for ann in ANNOTATORS:
    with open(f'{RESULTS_DIR}/{ann}/user_state.json') as f:
        state = json.load(f)
    annotator_data[ann] = {
        inst_id: parse_annotation(val)
        for inst_id, val in state['instance_id_to_label_to_value'].items()
    }

for ann, d in annotator_data.items():
    print(f'{ann}: {len(d)} instances annotated')

Loaded 589 instances from CSV
adde1214: 11 instances annotated
tejo9855: 23 instances annotated
umasgunturi@gmail.com: 23 instances annotated


## Helper Functions

In [5]:
SPAN1_COLOR = '#4e9af1'   # blue
SPAN2_COLOR = '#f4a432'   # orange
SPAN1_LABEL = 'Span 1'
SPAN2_LABEL = 'Span 2'

def render_text_with_spans(text, span1, span2):
    """Return HTML string with the two spans highlighted in the passage."""
    s1_start, s1_end = span1['start'], span1['end']
    s2_start, s2_end = span2['start'], span2['end']

    # Sort spans by position so we can insert tags left-to-right
    spans = sorted(
        [('span1', s1_start, s1_end), ('span2', s2_start, s2_end)],
        key=lambda x: x[1]
    )

    colors = {'span1': SPAN1_COLOR, 'span2': SPAN2_COLOR}
    labels = {'span1': SPAN1_LABEL, 'span2': SPAN2_LABEL}

    result = []
    cursor = 0
    for name, start, end in spans:
        result.append(text[cursor:start])
        color = colors[name]
        label = labels[name]
        token = text[start:end]
        result.append(
            f'<mark style="background:{color};color:#fff;padding:2px 5px;'
            f'border-radius:3px;font-weight:bold;" '
            f'title="{label}">{token}</mark>'
        )
        cursor = end
    result.append(text[cursor:])

    body = ''.join(result)
    body = body.replace('\n', '<br>')
    return body


def annotation_summary_html(ann_name, ann):
    """Return an HTML row summarising one annotator's response."""
    if ann is None:
        return f'<tr><td><b>{ann_name}</b></td><td colspan="4" style="color:#999">— not annotated —</td></tr>'

    s1 = '✅ Event' if ann.get('span1_is_event') else '❌ Not event'
    s2 = '✅ Event' if ann.get('span2_is_event') else '❌ Not event'

    if ann.get('span1_is_event') and ann.get('span2_is_event'):
        seq = ann.get('sequencing_rating', '—')
        caus = ann.get('causality_rating', '—')
    else:
        seq = caus = '—'

    return (
        f'<tr>'
        f'<td style="padding:4px 10px"><b>{ann_name}</b></td>'
        f'<td style="padding:4px 14px;color:{SPAN1_COLOR}">{s1}</td>'
        f'<td style="padding:4px 14px;color:{SPAN2_COLOR}">{s2}</td>'
        f'<td style="padding:4px 14px">{seq}</td>'
        f'<td style="padding:4px 14px">{caus}</td>'
        f'</tr>'
    )


def display_instance(inst_id, min_annotators=1):
    """Render one instance as rich HTML."""
    row = text_rows.get(inst_id)
    if row is None:
        print(f'No text found for {inst_id}')
        return

    # Collect annotators who have this instance
    anns = {ann: annotator_data[ann].get(inst_id) for ann in ANNOTATORS
            if inst_id in annotator_data[ann]}
    if len(anns) < min_annotators:
        return

    # Grab span info from any completed annotation (all instances share the same spans)
    sample_ann = next((a for a in anns.values() if a is not None), None)
    if sample_ann is None:
        return  # No completed annotation to pull span positions from

    span1 = sample_ann['span1']
    span2 = sample_ann['span2']

    text = row['sampled_text']
    highlighted = render_text_with_spans(text, span1, span2)

    # Legend
    legend = (
        f'<span style="background:{SPAN1_COLOR};color:#fff;padding:2px 7px;border-radius:3px;margin-right:6px">'
        f'{SPAN1_LABEL}: <b>{span1["token"]}</b></span>'
        f'<span style="background:{SPAN2_COLOR};color:#fff;padding:2px 7px;border-radius:3px">'
        f'{SPAN2_LABEL}: <b>{span2["token"]}</b></span>'
    )

    # Annotator table
    rows_html = ''.join(annotation_summary_html(ann, data) for ann, data in anns.items())
    table = (
        '<table style="border-collapse:collapse;margin-top:8px;font-size:0.92em">'
        '<thead><tr style="background:#f0f0f0">'
        '<th style="padding:4px 10px;text-align:left">Annotator</th>'
        f'<th style="padding:4px 14px;text-align:left;color:{SPAN1_COLOR}">Span 1 ({span1["token"]})</th>'
        f'<th style="padding:4px 14px;text-align:left;color:{SPAN2_COLOR}">Span 2 ({span2["token"]})</th>'
        '<th style="padding:4px 14px;text-align:left">Sequencing (1–5)</th>'
        '<th style="padding:4px 14px;text-align:left">Causality (1–5)</th>'
        '</tr></thead>'
        f'<tbody>{rows_html}</tbody>'
        '</table>'
    )

    source = row.get('source', '')
    doc_id = row.get('id', inst_id)

    html = (
        '<div style="border:1px solid #ddd;border-radius:6px;padding:14px 18px;margin-bottom:20px;font-family:sans-serif">'
        f'<div style="font-size:0.8em;color:#888;margin-bottom:6px">{inst_id} &nbsp;|&nbsp; {source} &nbsp;|&nbsp; {doc_id}</div>'
        f'<div style="margin-bottom:8px">{legend}</div>'
        f'<div style="background:#fafafa;border-left:3px solid #ccc;padding:10px 14px;font-size:0.93em;line-height:1.6;max-height:260px;overflow-y:auto">'
        f'{highlighted}</div>'
        f'{table}'
        '</div>'
    )
    display(HTML(html))

## Instances Annotated by All 4 Annotators

In [8]:
all_inst_sets = [set(annotator_data[ann].keys()) for ann in ANNOTATORS]
shared_all = set.intersection(*all_inst_sets)
print(f'{len(shared_all)} instances annotated by all 4 annotators')

for inst_id in sorted(shared_all):
    display_instance(inst_id)

10 instances annotated by all 4 annotators


Annotator,Span 1 (ducked),Span 2 (made),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,5,5
tejo9855,✅ Event,✅ Event,4,4
umasgunturi@gmail.com,✅ Event,✅ Event,5,4


Annotator,Span 1 (said),Span 2 (rainfall),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,❌ Not event,—,—
tejo9855,❌ Not event,✅ Event,—,—
umasgunturi@gmail.com,❌ Not event,✅ Event,—,—


Annotator,Span 1 (fly),Span 2 (attention),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,❌ Not event,—,—
tejo9855,✅ Event,❌ Not event,—,—
umasgunturi@gmail.com,✅ Event,❌ Not event,—,—


Annotator,Span 1 (heading),Span 2 (arraigned),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,4,1
tejo9855,❌ Not event,✅ Event,—,—
umasgunturi@gmail.com,✅ Event,✅ Event,5,3


...

Annotator,Span 1 (out),Span 2 (said),Sequencing (1–5),Causality (1–5)
adde1214,❌ Not event,❌ Not event,—,—
tejo9855,✅ Event,✅ Event,3,1
umasgunturi@gmail.com,✅ Event,✅ Event,5,4


Annotator,Span 1 (rises),Span 2 (following),Sequencing (1–5),Causality (1–5)
adde1214,❌ Not event,❌ Not event,—,—
tejo9855,❌ Not event,❌ Not event,—,—
umasgunturi@gmail.com,❌ Not event,❌ Not event,—,—


Annotator,Span 1 (ibid),Span 2 (ibid),Sequencing (1–5),Causality (1–5)
adde1214,❌ Not event,❌ Not event,—,—
tejo9855,❌ Not event,❌ Not event,—,—
umasgunturi@gmail.com,❌ Not event,❌ Not event,—,—


Annotator,Span 1 (onset),Span 2 (said),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,5,3
tejo9855,✅ Event,✅ Event,2,1
umasgunturi@gmail.com,❌ Not event,✅ Event,—,—


Annotator,Span 1 (regarding),Span 2 (add),Sequencing (1–5),Causality (1–5)
adde1214,❌ Not event,✅ Event,—,—
tejo9855,❌ Not event,❌ Not event,—,—
umasgunturi@gmail.com,❌ Not event,✅ Event,—,—


## All Annotated Instances (grouped by annotator)

In [9]:
for ann in ANNOTATORS:
    inst_ids = sorted(annotator_data[ann].keys())
    display(HTML(f'<h3 style="font-family:sans-serif;margin-top:24px">Annotator: {ann} ({len(inst_ids)} instances)</h3>'))
    for inst_id in inst_ids:
        display_instance(inst_id)

Annotator,Span 1 (ducked),Span 2 (made),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,5,5
tejo9855,✅ Event,✅ Event,4,4
umasgunturi@gmail.com,✅ Event,✅ Event,5,4


Annotator,Span 1 (said),Span 2 (rainfall),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,❌ Not event,—,—
tejo9855,❌ Not event,✅ Event,—,—
umasgunturi@gmail.com,❌ Not event,✅ Event,—,—


Annotator,Span 1 (fly),Span 2 (attention),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,❌ Not event,—,—
tejo9855,✅ Event,❌ Not event,—,—
umasgunturi@gmail.com,✅ Event,❌ Not event,—,—


Annotator,Span 1 (heading),Span 2 (arraigned),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,4,1
tejo9855,❌ Not event,✅ Event,—,—
umasgunturi@gmail.com,✅ Event,✅ Event,5,3


Annotator,Span 1 (created),Span 2 (relocated),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,4,1
tejo9855,✅ Event,✅ Event,3,2
umasgunturi@gmail.com,✅ Event,✅ Event,5,1


...

Annotator,Span 1 (onset),Span 2 (said),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,5,3
tejo9855,✅ Event,✅ Event,2,1
umasgunturi@gmail.com,❌ Not event,✅ Event,—,—


Annotator,Span 1 (regarding),Span 2 (add),Sequencing (1–5),Causality (1–5)
adde1214,❌ Not event,✅ Event,—,—
tejo9855,❌ Not event,❌ Not event,—,—
umasgunturi@gmail.com,❌ Not event,✅ Event,—,—


Annotator,Span 1 (visited),Span 2 (swept),Sequencing (1–5),Causality (1–5)
tejo9855,✅ Event,❌ Not event,—,—
umasgunturi@gmail.com,✅ Event,❌ Not event,—,—


Annotator,Span 1 (mistake),Span 2 (reported),Sequencing (1–5),Causality (1–5)
tejo9855,✅ Event,✅ Event,5,4
umasgunturi@gmail.com,✅ Event,✅ Event,5,5


Annotator,Span 1 (Read),Span 2 (Coordinating),Sequencing (1–5),Causality (1–5)
tejo9855,❌ Not event,❌ Not event,—,—
umasgunturi@gmail.com,❌ Not event,❌ Not event,—,—
